#### 0. Change directory

In [1]:
# Reference: https://stackoverflow.com/questions/15514593/importerror-no-module-named-when-trying-to-run-python-script/15622021#15622021
# Here choosing the folder that stores all the modelling modules
import sys
sys.path.append(r'S:\\ML_original_uncertainty\\LISFLOOD_FP\\Modelling')

#### 1. Packages

In [2]:
import json

from folder import *
from randomisation import random_values_generation
from dataPreparation import download_lidar, \
                            padding_combination, \
                            dem_raster_reference, \
                            terrain_shading             # For lidar downloading, padding calculation, dem reference generation, and terrain shading

from transformation import center_calculation, get_url, get_tile_files, \
                           MyPool, transformation_performance, transformation_parallelism # For transformation using multiprocessing

from rasterSimulation import raster_generation

from floodModel import tide_flow_data, bdy_generation, tideboundary_points, flood_simulation

# For creating plot
import matplotlib.pyplot

# For untransformation
from untransformation import untransformation_parallelism

import multiprocessing
from functools import partial


# For terrain shading
import os
import rioxarray as rxr
import xarray as xr
from osgeo import gdal

#### 2. Necessary variables

In [3]:
# # 20-m resolution
# # Construct a list of boundary
# resolution = 20
# number_pixel_x = 16 * 21
# number_pixel_y = 16 * 14

# xmin = 1767790
# ymin = 5471400

In [4]:
# # 10-m resolution
# # Construct a list of boundary
# resolution = 10
# number_pixel_x = 16 * 42
# number_pixel_y = 16 * 28

# xmin = 1767790
# ymin = 5471400

In [5]:
# 5-m resolution
# Construct a list of boundary
resolution = 5
number_pixel_x = 16 * 84
number_pixel_y = 16 * 56

xmin = 1767790
ymin = 5471400

In [6]:
# # 2-m resolution
# # Construct a list of boundary
# resolution = 2
# number_pixel_x = 16 * 210
# number_pixel_y = 16 * 140

# xmin = 1767790
# ymin = 5471400

##### 2.2. Other basic variables

In [7]:
xmax = xmin + resolution * number_pixel_x
ymax = ymin + resolution * number_pixel_y

# Boundary for the area of interest
boundary_1 = [xmin, ymin, xmax, ymax]

# Padding boundary for DEMs (use this boundary to create padding)
addition_2 = 16 * 3
boundary_2 = padding_combination(boundary_1, addition_2)

# Boundary for tiles (use this boundary to download LiDAR)
addition_3 = 16 * 3
boundary_3 = padding_combination(boundary_2, addition_3)

In [8]:
# LiDAR dataset name
lidar_name = "Wellington_2013"

# # Chunks and processors information (for computer has 8 cores)
# # For 10-m resolution
# size_of_chunk = 100
# size_of_processor = 4
# # For 20-m resolution
# size_of_chunk = 50
# size_of_processor = 2
# # For 2-m resolution
# size_of_chunk = 400
# size_of_processor = 6
# For 5-m resolution
size_of_chunk = 200
size_of_processor = 6

# Flow data
# flow_path = r"P:\Martin_publication\Grid_Orientation\Flow_multiple_events\\vers_002\\flow_5yevent_001.csv"
# flow_path = r"P:\Martin_publication\Grid_Orientation\Flow_multiple_events\\vers_002\\flow_10yevent_001.csv"
# flow_path = r"P:\Martin_publication\Grid_Orientation\Flow_multiple_events\\vers_002\\flow_20yevent_001.csv"
# flow_path = r"P:\Martin_publication\Grid_Orientation\Flow_multiple_events\\vers_002\\flow_50yevent_001.csv"
# flow_path = r"P:\Martin_publication\Grid_Orientation\Flow_multiple_events\\vers_002\\flow_1000yevent_001.csv"
flow_path = r"P:\Martin_publication\Grid_Orientation\Flow_Tide\\flow.csv"

# Tide data
tide_path = r"P:\\Martin_publication\\Grid_Orientation\\Flow_Tide\\tide.csv"

# date start and end
date_start = r"2005-01-05 00:00:00"
date_end = r"2005-01-07 00:00:00"

# Tide flow data
tide_flow_data(tide_path, flow_path, date_start, date_end)

# Extracting flowdepth rate (for flowdepth_extraction() function)
flowdepth_rate = 0

# Time to extract water (for water_extraction() function)
extract_wd = "out.max"
extract_wse = "out.mxe"
extract_elev = "elev"

##### 2.3. Preparing flood model parameters

In [9]:
# General parameters
general_para = {
    # GEOFABRICS -------------------
    # Land
    "land": 51559,
    # Bathymetry
    "bathymetry_contours": 50554,
    
    # Drains 
    "width": 5,
    
    # Osm
    "osm_id": 132793862, # 132793862 for Waikanae
    
    # River features
    "min_channel_width": 10,
    "max_channel_width": 120,
    "min_bank_height": 0.75,
    "max_bank_height": 2,
    
    # Others
    "channel_rec_id": 9253579,
    "channel_area_threshold": 121000000, #121000000 #19000000
    "width_centre_smoothing": 10,
    "cross_section_spacing": 10,
    "rec_alignment_tolerance": 65,
    "veg_lidar_classifications_to_keep": [2, 3, 4, 5, 9],
    
    # File
    "rec_file": "rec2_3.geojson",
    
    ## TRANSFORMATION
    'offset_x': 1700000,
    'offset_y': 5400000,
    
    # FLOOD MODEL -------------------
    # Tide points generation
    "tide_sampledis": 6, # 10
    "tide_num_points": 6, # 403
    
    # flood model paramters
    "saveint": 3600,
    "sim_time": 172800,
    
    # SEA FILTER
    'sea_catchment': 1
}

# Write out json
with open(fr"{other_data}/general_parameters.json", "w") as general_parameters_file:
    json.dump(general_para, general_parameters_file)

In [10]:
# Create tide_flow data
tide_flow_data(tide_path, flow_path, date_start, date_end)

# River points
river_points = (1774477.413, 5471521.558)

# Create BDY file
domain_name = "Waikanae"
bdy_generation(resolution, domain_name)

# Create tide boundary
tideboundary_points(boundary_1, False)

##### 2.4. Preparing some flood model inputs

In [11]:
# %%time
# # Download LiDAR
# download_lidar(boundary_3, lidar_name)

In [12]:
%%time
# Reference DEM without padding
dem_raster_reference(resolution, size_of_chunk, size_of_processor,
                     boundary_1, lidar_name, "no_padding")

Check files in dataset Wellington_2013
<Client: 'tcp://127.0.0.1:57404' processes=6 threads=6, memory=63.69 GiB>


Layer: 50554 is not `geometry_name`: GEOMETRY.


C:\Program Files\Anaconda3\envs\mluncertainty_001\lib\site-packages\geofabrics\geometry.py:221: ShapelyDeprecationWarning: The 'cascaded_union()' function is deprecated. Use 'unary_union()' instead.
  shapely.ops.cascaded_union(
C:\Program Files\Anaconda3\envs\mluncertainty_001\lib\site-packages\geofabrics\geometry.py:528: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  points = points.append(points_i)
C:\Program Files\Anaconda3\envs\mluncertainty_001\lib\site-packages\geofabrics\geometry.py:531: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  polygon = polygon.append(polygon_i)
C:\Program Files\Anaconda3\envs\mluncertainty_001\lib\site-packages\geofabrics\geometry.py:528: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  points = 

CPU times: total: 57 s
Wall time: 3min 14s


In [13]:
%%time
# Reference DEM with padding
dem_raster_reference(resolution, size_of_chunk, size_of_processor,
                     boundary_2, lidar_name, "padding")

Check files in dataset Wellington_2013
<Client: 'tcp://127.0.0.1:57700' processes=6 threads=6, memory=63.69 GiB>


Layer: 50554 is not `geometry_name`: GEOMETRY.


C:\Program Files\Anaconda3\envs\mluncertainty_001\lib\site-packages\geofabrics\geometry.py:221: ShapelyDeprecationWarning: The 'cascaded_union()' function is deprecated. Use 'unary_union()' instead.
  shapely.ops.cascaded_union(
C:\Program Files\Anaconda3\envs\mluncertainty_001\lib\site-packages\geofabrics\geometry.py:528: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  points = points.append(points_i)
C:\Program Files\Anaconda3\envs\mluncertainty_001\lib\site-packages\geofabrics\geometry.py:531: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  polygon = polygon.append(polygon_i)
C:\Program Files\Anaconda3\envs\mluncertainty_001\lib\site-packages\geofabrics\geometry.py:528: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  points = 

CPU times: total: 3min 10s
Wall time: 6min 39s


In [ ]:
# %%time
# # DEM roughness for hillshade/terrain shade
# dem_raster_reference(5, 200, 6,
#                      boundary_1, lidar_name, 'shading', True)

# # Terrain shading
# terrain_shading(50, 355)

In [14]:
# Calculate coordinates of center point
center_point = center_calculation(True)
center_x = center_point[0]                     # Extract x coordinate of center point
center_y = center_point[1]                     # Extract y coordinate of center point

# Get url list file
url_list_file = get_url(lidar_name)
tile_files = get_tile_files(lidar_name)

##### 2.5. Random transformations

##### 2.5.1. E translation (x)

In [ ]:
# # Transformation values
# ran_trans = random_values_generation(
#     1,
#     50,
#     [0, 0],
#     [0, resolution/2],
#     [0, 0],
#     'uniform',
#     True
# )

##### 2.5.2. N translation (y)

In [ ]:
# # Transformation values
# ran_trans = random_values_generation(
#     1,
#     50,
#     [0, 0],
#     [0, 0],
#     [0, resolution/2],
#     'uniform',
#     True
# )

##### 2.5.3. N-E translation

In [ ]:
# # Transformation values
# ran_trans = random_values_generation(
#     1,
#     50,
#     [0, 0],
#     [0, resolution/2],
#     [0, resolution/2],
#     'uniform',
#     True
# )

##### 2.5.4. Rotation

In [ ]:
# # Transformation values
# ran_trans = random_values_generation(
#     1,
#     50,
#     [0, 90],
#     [0, 0],
#     [0, 0],
#     'uniform',
#     True
# )

In [15]:
# Transformation values
ran_trans = random_values_generation(
    1,
    1,
    [0, 0],
    [0, 0],
    [0, 0],
    'uniform',
    True
)

In [ ]:
ran_trans

##### 2.5.5. N-E translation and rotation

In [ ]:
# # Transformation values
# ran_trans = random_values_generation(
#     1,
#     50,
#     [0, 90],
#     [0, resolution/2],
#     [0, resolution/2],
#     'uniform',
#     True
# )

In [ ]:
# ran_trans

##### 2.6. Only for 2-m resolution (if not skip this part)

In [ ]:
# # Transformation values
# ran_trans = random_values_generation(
#     1,
#     70,
#     [0, 90],
#     [0, resolution/2],
#     [0, resolution/2],
#     'uniform',
#     True
# )

In [ ]:
# import numpy as np
# ran_trans = np.delete(ran_trans, [3, 4, 12, 40, 41], 0)
# ran_trans = ran_trans[:50]

In [ ]:
# ran_trans

#### 3. Execution

In [16]:
%%time
transformation_parallelism(
    center_x,
    center_y,
    lidar_name,
    tile_files,
    30,
    url_list_file,
    ran_trans,
    1
)

CPU times: total: 15.6 ms
Wall time: 3min 50s


In [17]:
for ran in ran_trans:
    raster_generation(
        resolution, 
        size_of_chunk, size_of_processor,
        boundary_2, lidar_name,
        center_x, center_y,
        ran
    )

Check files in dataset Wellington_2013
<Client: 'tcp://127.0.0.1:58001' processes=6 threads=6, memory=63.69 GiB>


C:\Program Files\Anaconda3\envs\mluncertainty_001\lib\site-packages\geofabrics\geometry.py:221: ShapelyDeprecationWarning: The 'cascaded_union()' function is deprecated. Use 'unary_union()' instead.
  shapely.ops.cascaded_union(
C:\Program Files\Anaconda3\envs\mluncertainty_001\lib\site-packages\geofabrics\geometry.py:528: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  points = points.append(points_i)
C:\Program Files\Anaconda3\envs\mluncertainty_001\lib\site-packages\geofabrics\geometry.py:531: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  polygon = polygon.append(polygon_i)
C:\Program Files\Anaconda3\envs\mluncertainty_001\lib\site-packages\geofabrics\geometry.py:528: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  points = 

Check files in dataset Wellington_2013
<Client: 'tcp://127.0.0.1:50692' processes=6 threads=6, memory=63.69 GiB>


In [ ]:
# %%time
# # DEM CREATION
# # List parameters
# resolution_func = resolution
# chunk_size_func = size_of_chunk
# processor_func = size_of_processor
# padding_func = boundary_2
# lidar_dataset_name = lidar_name

# # Design func
# func = partial(
#     raster_generation,
#     resolution_func,
#     chunk_size_func,
#     processor_func,
#     padding_func,
#     lidar_dataset_name,
#     center_x, center_y
# )

# # Design pool
# pool = MyPool(1)
# pool.map(func, [ran for ran in ran_trans])
# pool.close()
# pool.join()

In [18]:
%%time
for ran_trans_i in ran_trans:
    flood_simulation(
        domain_name,
        river_points,
        center_x, center_y,
        fr"S:\\new_versions\\data",
        ran_trans_i
    )

CPU times: total: 31.2 ms
Wall time: 3h 56min 35s


In [19]:
%%time
# Water depth
untransformation_parallelism(
    extract_wd,
    ran_trans,
    1
)

CPU times: total: 0 ns
Wall time: 2min 24s


In [20]:
%%time
# Water surface elevation
untransformation_parallelism(
    extract_wse,
    ran_trans,
    1
)

CPU times: total: 15.6 ms
Wall time: 2min 22s


In [21]:
%%time
# Elevation
untransformation_parallelism(
    extract_elev,
    ran_trans,
    1
)

CPU times: total: 15.6 ms
Wall time: 2min 31s
